# Customer Segmentation Analysis & Dashboard Upload

This notebook provides a complete walk-through of customer segmentation using K-Means clustering, data visualization, and programmatic upload to the Customer Segmentation Analytics Dashboard.

## 1. Import Dependencies & Load Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import requests

# Set plot styles
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Load the sample dataset
filepath = 'sample_data/customers.csv'
if not os.path.exists(filepath):
    # Fallback to absolute path or download from public folder
    df = pd.read_csv('https://raw.githubusercontent.com/datasets/customer-segmentation/master/customers.csv')
else:
    df = pd.read_csv(filepath)

print(f"Dataset loaded successfully. Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Check dataset statistics
df.describe()

In [ ]:
# Gender distribution
sns.countplot(data=df, x='Gender', palette='pastel')
plt.title('Customer Gender Distribution')
plt.show()

In [ ]:
# Income vs Spending Score Distribution
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Gender', alpha=0.8)
plt.title('Annual Income vs Spending Score')
plt.show()

## 3. K-Means Clustering

In [ ]:
# Standardise features
features = df[['Annual Income (k$)', 'Spending Score (1-100)']]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Run K-Means with 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(scaled_features)
df.head()

In [ ]:
# Plot cluster results
sns.scatterplot(
    data=df, 
    x='Annual Income (k$)', 
    y='Spending Score (1-100)', 
    hue='Cluster', 
    palette='viridis', 
    s=100, 
    alpha=0.8
)
plt.title('Customer Segments (K-Means Clustering)')
plt.show()

## 4. Programmatic Dataset Upload to Dashboard

Now we will upload this customer dataset to the Flask API backend of the Customer Segmentation Dashboard so you can view the fully interactive visualization in the web application.

In [ ]:
url = 'http://localhost:5000/api/upload'
csv_filepath = 'sample_data/customers.csv'

if os.path.exists(csv_filepath):
    print(f"[*] Sending '{csv_filepath}' to dashboard API at {url}...")
    with open(csv_filepath, 'rb') as f:
        files = {'file': (os.path.basename(csv_filepath), f, 'text/csv')}
        try:
            response = requests.post(url, files=files)
            if response.status_code == 200:
                print("[+] Upload successful! The dashboard has been populated.")
                print(response.json())
            else:
                print(f"[-] Error: Status {response.status_code}. Detail: {response.text}")
        except requests.exceptions.ConnectionError:
            print("[-] Connection Error: Is your Flask backend running on http://localhost:5000?")
else:
    print(f"[-] File '{csv_filepath}' not found. Run generate_sample_data.py first!")